# Demo 3: Interpretability of SigmaSelectorAnalyze how the SigmaSelector routes LocationEncoder branches across different$\sigma$ (frequency scales).**Architecture note:** SigmaSelector takes _geographic coordinates_ as input(via Equal Earth projection), not image features. It learns a spatial prior --certain regions of the globe benefit more from fine-grained vs. coarse-grainedpositional encodings. This is useful for understanding latitude-dependentdistortion patterns.We compare urban vs. natural regions to see if the model has learned differentrouting strategies for different parts of the world.

In [ ]:
import sysfrom pathlib import Pathimport torchimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom PIL import Image_root = Path.cwd().parent if Path.cwd().name == 'demos' else Path.cwd()sys.path.insert(0, str(_root))from demos.demo_utils import load_geotx_model, STREETVIEW_CSV, STREETVIEW_IMAGESfrom geoclip.train.dataloader import img_val_transform%matplotlib inline

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'model = load_geotx_model(device)# Sigma values from the LocationEncodersigmas = model.location_encoder.sigma  # [2^0, 2^4, 2^8] = [1, 16, 256]print(f'Sigma values: {sigmas}')print(f'SigmaSelector enabled: {model.location_encoder.use_sigma_selector}')

In [ ]:
# --- Curated example selection ---# We define geographic bounding boxes for extreme urban and natural areas,# then sample images from each region.df_all = pd.read_csv(STREETVIEW_CSV)URBAN_REGIONS = {    'New York City': (40.5, 40.9, -74.1, -73.7),    'Tokyo': (35.5, 35.8, 139.5, 139.9),    'London': (51.4, 51.6, -0.3, 0.1),    'Singapore': (1.25, 1.45, 103.6, 104.0),}NATURAL_REGIONS = {    'Iceland (rural)': (63.5, 66.5, -23.0, -13.0),    'Norway (fjords)': (59.0, 62.0, 5.0, 11.0),    'Finland (forests)': (60.0, 65.0, 24.0, 31.0),    'New Zealand (South Isl.)': (-46.0, -43.0, 168.0, 173.0),}def sample_images_for_regions(region_dict, n_per_region=2, seed=42):    rng = np.random.RandomState(seed)    samples = []    for name, (lat_lo, lat_hi, lon_lo, lon_hi) in region_dict.items():        mask = (            (df_all['LAT'] >= lat_lo) & (df_all['LAT'] <= lat_hi)            & (df_all['LON'] >= lon_lo) & (df_all['LON'] <= lon_hi)        )        region_df = df_all[mask]        if len(region_df) == 0:            print(f'  Warning: no images found in {name}')            continue        n_sample = min(n_per_region, len(region_df))        idx = rng.choice(region_df.index, n_sample, replace=False)        for i in idx:            samples.append({                'name': name,                'image_file': region_df.loc[i, 'IMG_FILE'],                'lat': region_df.loc[i, 'LAT'],                'lon': region_df.loc[i, 'LON'],            })    return samplesurban_samples = sample_images_for_regions(URBAN_REGIONS)natural_samples = sample_images_for_regions(NATURAL_REGIONS)all_samples = urban_samples + natural_samplesprint(f'Selected {len(urban_samples)} urban + {len(natural_samples)} natural = {len(all_samples)} images')

In [ ]:
# --- Extract SigmaSelector weights for each image ---transform = img_val_transform()results = []for sample in all_samples:    img_path = STREETVIEW_IMAGES / sample['image_file']    if not img_path.exists():        print(f'  Missing: {img_path}')        continue        # Preprocess image    img = Image.open(img_path).convert('RGB')    img_tensor = transform(img).unsqueeze(0).to(device)        # GPS for the image (SigmaSelector takes GPS as input)    gps = torch.tensor([[sample['lat'], sample['lon']]], dtype=torch.float32).to(device)        # Get sigma weights    with torch.no_grad():        weights = model.location_encoder.get_sigma_weights(gps).cpu().numpy().flatten()        results.append({        **sample,        'image': img,        'weights': weights,    })print(f'Extracted weights for {len(results)} images')

In [ ]:
# --- Side-by-side visualization ---
n_cols = 4
n_rows = (len(results) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows * 2, n_cols,
    figsize=(n_cols * 4, n_rows * 6),
    gridspec_kw={'height_ratios': [2, 1] * n_rows},
)

sigma_labels = [f'$\\sigma$={s} km' for s in sigmas]
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, res in enumerate(results):
    row = (idx // n_cols) * 2
    col = idx % n_cols

    ax_img = axes[row, col] if n_rows > 1 else axes[col]
    ax_bar = axes[row + 1, col] if n_rows > 1 else axes[col]

    # Show image
    ax_img.imshow(res['image'])
    img_title = res['name'] + '' + f'({res["lat"]:.2f}, {res["lon"]:.2f})'
    ax_img.set_title(img_title, fontsize=9)
    ax_img.axis('off')

    # Show bar chart of sigma weights
    bars = ax_bar.bar(range(len(sigmas)), res['weights'], color=colors, edgecolor='black', linewidth=0.5)
    ax_bar.set_xticks(range(len(sigmas)))
    ax_bar.set_xticklabels(sigma_labels, fontsize=8)
    ax_bar.set_ylim(0, 1.0)
    ax_bar.set_ylabel('Weight', fontsize=8)

    # Annotate bar values
    for bar, w in zip(bars, res['weights']):
        ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{w:.3f}', ha='center', fontsize=7)

# Hide unused axes
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else np.atleast_1d(axes)
for j in range(len(results) * 2, len(axes_flat)):
    axes_flat[j].axis('off')

suptitle_line1 = 'SigmaSelector Routing Weights: Urban (top 2 rows) vs Natural (bottom 2 rows)'
suptitle_line2 = 'Higher $\\sigma$ = fine details | Lower $\\sigma$ = global patterns'
fig.suptitle(suptitle_line1 + '' + suptitle_line2, fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# --- Summary: average weights by category ---urban_weights = np.array([r['weights'] for r in results if r['name'] in URBAN_REGIONS])natural_weights = np.array([r['weights'] for r in results if r['name'] in NATURAL_REGIONS])fig, ax = plt.subplots(figsize=(8, 5))x = np.arange(len(sigmas))width = 0.35if len(urban_weights) > 0:    ax.bar(x - width/2, urban_weights.mean(axis=0), width,           label='Urban', color='#e74c3c', edgecolor='black', yerr=urban_weights.std(axis=0))if len(natural_weights) > 0:    ax.bar(x + width/2, natural_weights.mean(axis=0), width,           label='Natural', color='#2ecc71', edgecolor='black', yerr=natural_weights.std(axis=0))ax.set_xticks(x)ax.set_xticklabels(sigma_labels, fontsize=10)ax.set_ylabel('Mean Routing Weight', fontsize=11)ax.set_ylim(0, 1.0)ax.legend(fontsize=10)ax.set_title('Average SigmaSelector Weights: Urban vs Natural Landscapes', fontsize=12)ax.grid(axis='y', alpha=0.3)fig.tight_layout()plt.show()

In [ ]:
# --- Latitude-band analysis: a more direct test of SigmaSelector ---
# Since SigmaSelector operates on (projected) coordinates, the most natural
# hypothesis is that routing varies by latitude (Equal Earth projection
# distortion increases toward the poles).

lat_bands = {
    'Equatorial (0-30)': (0, 30),
    'Mid-latitude (30-55)': (30, 55),
    'High-latitude (55-90)': (55, 90),
}

band_weights = {}
for band_name, (lat_lo, lat_hi) in lat_bands.items():
    band_results = [r for r in results if lat_lo <= abs(r['lat']) < lat_hi]
    if band_results:
        w = np.array([r['weights'] for r in band_results])
        band_weights[band_name] = (w.mean(axis=0), w.std(axis=0))
        print(f'{band_name}: {len(band_results)} images')
    else:
        print(f'{band_name}: NO images')

if band_weights:
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(sigmas))
    width = 0.25
    colors_band = ['#e74c3c', '#f39c12', '#3498db']

    for i, (band_name, (mean_w, std_w)) in enumerate(band_weights.items()):
        ax.bar(x + i * width, mean_w, width,
               label=band_name, color=colors_band[i],
               edgecolor='black', yerr=std_w)

    ax.set_xticks(x + width)
    ax.set_xticklabels(sigma_labels, fontsize=10)
    ax.set_ylabel('Mean Routing Weight', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)
    band_title = ('SigmaSelector Weights by Latitude Band'
                  '(Routing depends on projected coordinates, not image content)')
    ax.set_title(band_title, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    plt.show()